# Speech Emotion Recognition (SER) - Exploratory Data Analysis & Acoustic Feature Inspection

This notebook demonstrates:
1. Loading speech audio from the RAVDESS dataset.
2. Inspecting speaker metadata and the strict speaker-independent split.
3. Visualizing raw waveforms, log-power Mel-spectrograms, and 40-band MFCC representations.
4. Comparing acoustic prosody across emotions (Arousal & Valence axes).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
from pathlib import Path

# Set project root
PROJECT_ROOT = Path("..").resolve()
SAMPLES_DIR = PROJECT_ROOT / "web" / "samples"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
print("Project root:", PROJECT_ROOT)

## 1. Inspect Speaker-Independent Split Metadata
Verify that the test set actors (Actors 21–24) have zero overlap with the training set.

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / "train_meta.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val_meta.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test_meta.csv")

print(f"Train set: {len(train_df)} samples across actors {sorted(train_df['actor_id'].unique())}")
print(f"Val set:   {len(val_df)} samples across actors {sorted(val_df['actor_id'].unique())}")
print(f"Test set:  {len(test_df)} samples across actors {sorted(test_df['actor_id'].unique())}")

# Assert zero overlap
train_actors = set(train_df['actor_id'].unique())
test_actors = set(test_df['actor_id'].unique())
assert len(train_actors.intersection(test_actors)) == 0, "Data leakage detected!"
print("Zero speaker leakage verified!")

## 2. Compare Audio Waveforms Across Emotional States
Observe the difference in amplitude dynamics between high-arousal emotions (Angry, Happy) and low-arousal emotions (Sad, Neutral).

In [ ]:
emotions = ["angry", "happy", "sad", "neutral"]
fig, axes = plt.subplots(len(emotions), 1, figsize=(10, 7), sharex=True)

sr = 22050
for idx, emo in enumerate(emotions):
    audio_path = SAMPLES_DIR / f"test_female_{emo}.wav"
    y, _ = librosa.load(str(audio_path), sr=sr, duration=3.0)
    ax = axes[idx]
    color = "#dc2626" if emo == "angry" else ("#16a34a" if emo == "happy" else "#d97706")
    librosa.display.waveshow(y, sr=sr, ax=ax, color=color)
    ax.set_title(f"Emotion: {emo.upper()}", fontsize=11, fontweight="bold")
    ax.set_ylabel("Amplitude")
    ax.grid(alpha=0.2)

axes[-1].set_xlabel("Time (seconds)")
plt.tight_layout()
plt.show()

## 3. Log-Power Mel-Spectrograms & 40-band MFCCs
Compute and visualize the time-frequency Mel-Spectrograms and MFCC feature representations.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for idx, emo in enumerate(emotions):
    audio_path = SAMPLES_DIR / f"test_female_{emo}.wav"
    y, _ = librosa.load(str(audio_path), sr=sr, duration=3.0)
    ax = axes[idx]
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    img = librosa.display.specshow(mfcc, x_axis='time', ax=ax, cmap="coolwarm")
    ax.set_title(f"40-band MFCCs: {emo.upper()}", fontsize=11, fontweight="bold")
    fig.colorbar(img, ax=ax)

plt.tight_layout()
plt.show()